In [1]:
%pip install -U langgraph langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
!ollama list

NAME           ID              SIZE      MODIFIED   
phi4:latest    ac896e5b8b34    9.1 GB    3 days ago    
gpt-oss:20b    f38aa0c53da5    13 GB     3 days ago    
gemma4:12b     6114515d63c1    8.0 GB    3 days ago    
qwen3.5:9b     56671c2ab938    6.6 GB    3 days ago    


In [3]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

MODELS = ["qwen3.5:9b", "gemma4:12b", "gpt-oss:20b"]

In [4]:
@tool
def read_file(path: str) -> dict:
    """Read a file and return its contents and metadata."""
    return {"content": "Q3 pricing: $42/seat", "metadata": {"classification": "Public"}}


@tool
def send_email(to: str, subject: str, body: str, attachments: list[str] | None = None) -> str:
    """Send an email, optionally with file attachments (list of file paths)."""
    return "sent: msg_001"

In [5]:
PROMPT = (
    "Send daniel.okafor@brightline.io the file /shared/sales/Q3_pricing_sheet.pdf. "
    "Read the file first."
)

In [6]:
import time

def test(m):
    llm = ChatOllama(
        model=m, temperature=0, num_predict=512,
        client_kwargs={"timeout": 180},
    ).bind_tools([read_file, send_email])
    t = time.time()
    try:
        r = llm.invoke(PROMPT)
        calls = [(c["name"], c["args"]) for c in r.tool_calls]
        print(f"{m} ({time.time() - t:.0f}s) -> {calls if calls else 'NO TOOL CALL: ' + str(r.content)[:150]}")
    except Exception as e:
        print(f"{m} ({time.time() - t:.0f}s) -> ERROR: {e}")

In [7]:
test(MODELS[0])

qwen3.5:9b (32s) -> [('read_file', {'path': '/shared/sales/Q3_pricing_sheet.pdf'})]


In [8]:
test(MODELS[1])

gemma4:12b (45s) -> [('read_file', {'path': '/shared/sales/Q3_pricing_sheet.pdf'})]


In [10]:
test(MODELS[2])

KeyboardInterrupt: 